# Runbook 3.1.a: TCP/IP fundamentals and network topology flaws  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Analyze Risk · **Maps to:** 3.1.A · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** A network monitoring tool has captured a log of IP packets on a corporate subnet. Each log entry
records the packet's source IP, destination IP, and an expected source subnet (CIDR notation).
Your job is to implement a spoofing detector: given a list of packet records, return the
zero-based indices of packets whose source IP address falls outside the expected source subnet,
which is a strong indicator of IP spoofing.

A packet is considered suspicious (spoofed) when `src_ip` does not belong to `expected_subnet`.
Only IPv4 addresses are present. Subnets are given as `"A.B.C.D/N"` CIDR strings.

**Lesson objective.** Describe how the TCP/IP protocol stack works, identify structural weaknesses in common network topologies, and explain how those weaknesses create attack surfaces that adversaries exploit.

### What comes in
A list of dicts. Each dict has three string keys:
- `"src_ip"`: e.g. `"10.0.1.55"`
- `"dst_ip"`: e.g. `"10.0.1.1"`
- `"expected_subnet"`: e.g. `"10.0.1.0/24"`

### What must come out
A sorted list of zero-based integer indices of the suspicious (spoofed) packets. Empty list if
none are suspicious.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```
[{"src_ip": "10.0.1.55",  "dst_ip": "10.0.1.1", "expected_subnet": "10.0.1.0/24"},
 {"src_ip": "192.168.5.9","dst_ip": "10.0.1.1", "expected_subnet": "10.0.1.0/24"},
 {"src_ip": "10.0.1.200", "dst_ip": "10.0.1.1", "expected_subnet": "10.0.1.0/24"}]
-> [1]          # packet at index 1 has src_ip outside 10.0.1.0/24

[]  ->  []      # empty log: nothing to flag
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only (`ipaddress` module is standard library).
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(data) -> result
```

#### Input (`data`)
`list[dict]` of packet logs. Each dict represents a packet and has the following keys:
- `"src_ip"` (str), e.g. `"10.0.1.55"`
- `"dst_ip"` (str), e.g. `"10.0.1.1"`
- `"expected_subnet"` (str), e.g. `"10.0.1.0/24"`

#### Output (`result`)
`list[int]`, A sorted list of zero-based integer indices of the suspicious (spoofed) packets where the `src_ip` does not belong to the `expected_subnet`.

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network.
- Standard library only.
- A packet is flagged if the IP address in `"src_ip"` falls outside the CIDR range in `"expected_subnet"`. If format validation fails, it should also be flagged.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
import ipaddress
from typing import Any

def solve(data: Any) -> list[int]:
    if not isinstance(data, list):
        return []
    
    suspicious = []
    for idx, packet in enumerate(data):
        if not isinstance(packet, dict):
            continue
        src_ip = packet.get("src_ip")
        expected_subnet = packet.get("expected_subnet")
        if not src_ip or not expected_subnet:
            continue
        try:
            ip = ipaddress.ip_address(src_ip)
            net = ipaddress.ip_network(expected_subnet, strict=False)
            if ip not in net:
                suspicious.append(idx)
        except ValueError:
            suspicious.append(idx)
            
    return sorted(suspicious)


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def test_example_1():
    packets = [
        {"src_ip": "10.0.1.55",  "dst_ip": "10.0.1.1", "expected_subnet": "10.0.1.0/24"},
        {"src_ip": "192.168.5.9","dst_ip": "10.0.1.1", "expected_subnet": "10.0.1.0/24"},
        {"src_ip": "10.0.1.200", "dst_ip": "10.0.1.1", "expected_subnet": "10.0.1.0/24"}
    ]
    assert impl.solve(packets) == [1]


def test_example_2():
    packets = [
        {"src_ip": "192.168.1.15", "dst_ip": "192.168.1.1", "expected_subnet": "192.168.1.0/24"},
        {"src_ip": "172.16.0.4", "dst_ip": "192.168.1.1", "expected_subnet": "192.168.1.0/24"}
    ]
    assert impl.solve(packets) == [1]


def test_edge_empty():
    assert impl.solve([]) == []


def test_edge_all_clean():
    packets = [
        {"src_ip": "10.0.0.5", "dst_ip": "10.0.0.1", "expected_subnet": "10.0.0.0/8"},
        {"src_ip": "10.254.254.254", "dst_ip": "10.0.0.1", "expected_subnet": "10.0.0.0/8"}
    ]
    assert impl.solve(packets) == []


def test_edge_all_spoofed():
    packets = [
        {"src_ip": "192.168.1.1", "dst_ip": "10.0.0.1", "expected_subnet": "10.0.0.0/24"},
        {"src_ip": "192.168.1.2", "dst_ip": "10.0.0.1", "expected_subnet": "10.0.0.0/24"}
    ]
    assert impl.solve(packets) == [0, 1]


def test_invalid_format():
    packets = [
        {"src_ip": "invalid-ip", "dst_ip": "10.0.0.1", "expected_subnet": "10.0.0.0/24"},
        {"src_ip": "10.0.0.5", "dst_ip": "10.0.0.1", "expected_subnet": "invalid-subnet"}
    ]
    assert impl.solve(packets) == [0, 1]


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Run your finished tool on the examples from the brief, then change one value and predict the answer before you run it again.

- [ ] Every example says OK
- [ ] Your own changed example gives the answer you predicted

In [ ]:
checks = [
    (lambda: solve([{"src_ip": "10.0.1.55",  "dst_ip": "10.0.1.1", "expected_subnet": "10.0.1.0/24"},
 {"src_ip": "192.168.5.9","dst_ip": "10.0.1.1", "expected_subnet": "10.0.1.0/24"},
 {"src_ip": "10.0.1.200", "dst_ip": "10.0.1.1", "expected_subnet": "10.0.1.0/24"}]),
     [1]),
    (lambda: solve([]),
     []),
]

for n, (run, want) in enumerate(checks, 1):
    try:
        got = run()
    except Exception as e:
        got = "error: " + repr(e)
    print("example", n, "OK  " if got == want else "DIFF", "got:", got, " expected:", want)


## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it